# Preprocessing Pipeline — CrisisMMD Dataset

This notebook performs data cleaning and label reconciliation for the CrisisMMD multimodal dataset. It prepares the annotations for both unimodal and multimodal learning pipelines.

We load the unified annotations (with image hashes), handle noisy/duplicate entries, merge rare classes, and generate fused labels for `mm_info` and `mm_human`.

## Imports and setup

In [1]:
import os
import pandas as pd

import re
import html
import emoji
import unicodedata

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Load paths from .env file
data_dir = os.getenv("DATA_DIR")

# Load the unified TSV file with image hashes
annotations_file = os.path.join(data_dir, "annotations_with_hash.tsv")

In [4]:
df = pd.read_csv(annotations_file, sep="\t")

In [5]:
df.head()

,tweet_id,image_id,text_info,text_info_conf,image_info,image_info_conf,text_human,text_human_conf,image_human,image_human_conf,...,image_damage_conf,tweet_text,image_url,image_path,disaster_type,general_disaster_type,img_width,img_height,img_aspect_ratio,img_hash_str
0,917791044158185473,917791044158185473_0,informative,1.0000,informative,0.6766,other_relevant_information,1.0000,other_relevant_information,0.6766,...,NaN,RT @Gizmodo: Wildfires raging through Northern...,http://pbs.twimg.com/media/DLyi_WYVYAApwNg.jpg,data_image/california_wildfires/10_10_2017/917...,california_wildfires,wildfire,800,450,1.777778,86c476bb39c2b16c
1,917791130590183424,917791130590183424_0,informative,1.0000,informative,0.6667,infrastructure_and_utility_damage,1.0000,affected_individuals,0.6667,...,NaN,PHOTOS: Deadly wildfires rage in California ht...,http://pbs.twimg.com/media/DLymKm9UMAAu0qw.jpg,data_image/california_wildfires/10_10_2017/917...,california_wildfires,wildfire,1200,677,1.772526,f28e9aa98b96a730
2,917791291823591425,917791291823591425_0,informative,0.6813,informative,1.0000,other_relevant_information,0.6813,infrastructure_and_utility_damage,1.0000,...,1.0,RT @Cal_OES: PLS SHARE: We're capturing wildfi...,http://pbs.twimg.com/media/DLudaaZV4AAjT7x.jpg,data_image/california_wildfires/10_10_2017/917...,california_wildfires,wildfire,640,480,1.333333,a9bdb18391838bba
3,917791291823591425,917791291823591425_1,informative,0.6813,not_informative,1.0000,other_relevant_information,0.6813,not_humanitarian,1.0000,...,NaN,RT @Cal_OES: PLS SHARE: We're capturing wildfi...,http://pbs.twimg.com/media/DLudaZXUMAABAEZ.jpg,data_image/california_wildfires/10_10_2017/917...,california_wildfires,wildfire,1200,900,1.333333,b582566fab45cac8
4,917792092100988929,917792092100988929_0,informative,0.6727,informative,0.6612,other_relevant_information,0.6727,infrastructure_and_utility_damage,0.6612,...,1.0,RT @TIME: California's raging wildfires as you...,http://pbs.twimg.com/media/DLwNe-NXUAE0XCw.jpg,data_image/california_wildfires/10_10_2017/917...,california_wildfires,wildfire,600,400,1.500000,86826be7394ed43a


## Resolve duplicated tweet entries

Multiple entries may share the same `tweet_id` (i.e., same tweet with multiple images). We reconcile their labels using a custom confidence-aware strategy:

- For each label, we compute a **weighted score** that balances **frequency** and **mean confidence**:
```math
\text{score}(L) = \alpha \cdot \text{frequency}(L) + (1 - \alpha) \cdot \text{average\_confidence}(L)
```
Where $\alpha$ is a tunable parameter:

$\alpha = 1$ → pure frequency

$\alpha = 0$ → pure confidence

- The label with the highest weighted score is selected





In [6]:
# Show a few examples of duplicated tweet_id entries
print("Sample duplicated tweet_id entries (same tweet, different images):")
sample_tweet_dups = df[df["tweet_id"].duplicated(keep=False)].sort_values("tweet_id").head(10)
display(sample_tweet_dups[["tweet_id", "image_id", "tweet_text", "text_info", "text_info_conf", "text_human", "text_human_conf"]])


Sample duplicated tweet_id entries (same tweet, different images):


,tweet_id,image_id,tweet_text,text_info,text_info_conf,text_human,text_human_conf
17072,869972354004393987,869972354004393987_2,Pak Navy continues Humanitarian Assistance and...,informative,1.0000,rescue_volunteering_or_donation_effort,1.0000
17073,869972354004393987,869972354004393987_3,Pak Navy continues Humanitarian Assistance and...,informative,1.0000,rescue_volunteering_or_donation_effort,1.0000
17070,869972354004393987,869972354004393987_0,Pak Navy continues Humanitarian Assistance and...,informative,1.0000,rescue_volunteering_or_donation_effort,1.0000
17071,869972354004393987,869972354004393987_1,Pak Navy continues Humanitarian Assistance and...,informative,1.0000,rescue_volunteering_or_donation_effort,1.0000
17074,869977622377320448,869977622377320448_0,RT @DisastersChart: #Sentinel2 was used to map...,informative,0.7809,other_relevant_information,0.7809
17076,869977622377320448,869977622377320448_2,RT @DisastersChart: #Sentinel2 was used to map...,informative,0.7809,other_relevant_information,0.7809
17075,869977622377320448,869977622377320448_1,RT @DisastersChart: #Sentinel2 was used to map...,informative,0.7809,other_relevant_information,0.7809
17093,870008928054259712,870008928054259712_0,"@Anchoveebrother Hello dear Chile birds, Marti...",not_informative,1.0000,not_humanitarian,1.0000
17095,870008928054259712,870008928054259712_2,"@Anchoveebrother Hello dear Chile birds, Marti...",not_informative,1.0000,not_humanitarian,1.0000
17094,870008928054259712,870008928054259712_1,"@Anchoveebrother Hello dear Chile birds, Marti...",not_informative,1.0000,not_humanitarian,1.0000


In [7]:
# Mark only the second and subsequent occurrences of each tweet_id as duplicate
df["text_dup"] = df.duplicated(subset=["tweet_id"], keep="first")

In [8]:
duplicated_tweets = df[df.duplicated(subset=["tweet_id"], keep=False)].copy()

# Find tweet_id groups with inconsistent text_info or text_human
conflicting_tweet_ids = (
	duplicated_tweets.groupby("tweet_id")[["text_info", "text_human"]]
	.nunique()
	.query("text_info > 1 or text_human > 1")
	.index
)

print(f"⚠️ Tweet IDs with conflicting text labels: {len(conflicting_tweet_ids)}")

# Show a few examples
df_conflicting_tweets = df[df["tweet_id"].isin(conflicting_tweet_ids)].sort_values("tweet_id")
display(df_conflicting_tweets[["tweet_id", "tweet_text", "text_info", "text_info_conf", "text_human", "text_human_conf"]].head(10))


⚠️ Tweet IDs with conflicting text labels: 40


,tweet_id,tweet_text,text_info,text_info_conf,text_human,text_human_conf
17492,872235889820291072,"Accumulating hail in Cleveland, NM [Mora Co.]....",informative,0.6822,other_relevant_information,0.6822
17491,872235889820291072,"Accumulating hail in Cleveland, NM [Mora Co.]....",not_informative,0.6667,not_humanitarian,0.6667
17550,872672448394780672,#Srilanka #Floods Seen From #Space https://t.c...,not_informative,0.6818,not_humanitarian,0.6818
17549,872672448394780672,#Srilanka #Floods Seen From #Space https://t.c...,informative,0.7478,other_relevant_information,0.7478
1595,901646123080830976,RT @yIIeza: When we get back to SCHS after Har...,not_informative,1.0000,not_humanitarian,1.0000
1596,901646123080830976,RT @yIIeza: When we get back to SCHS after Har...,not_informative,1.0000,not_humanitarian,1.0000
1597,901646123080830976,RT @yIIeza: When we get back to SCHS after Har...,informative,0.6814,other_relevant_information,0.6814
1594,901646123080830976,RT @yIIeza: When we get back to SCHS after Har...,not_informative,1.0000,not_humanitarian,1.0000
3431,905491119705702400,"PHOTOS: New album ""Texas recovers from Harvey""...",informative,1.0000,other_relevant_information,1.0000
3432,905491119705702400,"PHOTOS: New album ""Texas recovers from Harvey""...",informative,1.0000,other_relevant_information,1.0000


In [9]:
def print_class_distribution(df, label_col):
    print(f"\n`{label_col}` distribution:")
    counts = df[label_col].value_counts(dropna=False)
    percentages = df[label_col].value_counts(normalize=True, dropna=False) * 100
    print(pd.DataFrame({'count': counts, 'percent': percentages.round(2)}))

In [10]:
print_class_distribution(df, "text_info")
print_class_distribution(df, "text_human")


`text_info` distribution:
                 count  percent
text_info                      
informative      12855    71.09
not_informative   5227    28.91

`text_human` distribution:
                                        count  percent
text_human                                            
other_relevant_information               6499    35.94
not_humanitarian                         5227    28.91
rescue_volunteering_or_donation_effort   3776    20.88
infrastructure_and_utility_damage        1428     7.90
injured_or_dead_people                    533     2.95
affected_individuals                      517     2.86
vehicle_damage                             61     0.34
missing_or_found_people                    41     0.23


In [11]:
def reconcile_with_confidence(group, label_col, conf_col, alpha=0.6):
    """Reconcile a label for a group of duplicated entries using both frequency and confidence."""
    scores = {}
    total = len(group)
    for label in group[label_col].unique():
        label_group = group[group[label_col] == label]
        freq = len(label_group) / total
        avg_conf = label_group[conf_col].mean()
        scores[label] = alpha * freq + (1 - alpha) * avg_conf
    return max(scores, key=scores.get)


In [12]:
for label_col in ["text_info", "text_human"]:
    conf_col = label_col + "_conf"
    reconciled = (
        duplicated_tweets.groupby("tweet_id", group_keys=False)
        .apply(lambda g: reconcile_with_confidence(g, label_col, conf_col, alpha=0.6), include_groups=False)
    )
    # Map reconciled labels back into the full dataframe
    df.loc[df["tweet_id"].isin(reconciled.index), label_col] = df["tweet_id"].map(reconciled)


In [13]:
# Tweet text duplication stats
n_total = len(df)
n_dup_tweets = df["text_dup"].sum()
n_unique_conflicting_tweet_ids = len(conflicting_tweet_ids)

print(f"Total duplicated tweet_id rows (excluding first occurrence): {n_dup_tweets}")
print(f"Unique tweet_id groups reconciled: {n_unique_conflicting_tweet_ids}")


Total duplicated tweet_id rows (excluding first occurrence): 2024
Unique tweet_id groups reconciled: 40


In [14]:
print_class_distribution(df, "text_info")
print_class_distribution(df, "text_human")


`text_info` distribution:
                 count  percent
text_info                      
informative      12854    71.09
not_informative   5228    28.91

`text_human` distribution:
                                        count  percent
text_human                                            
other_relevant_information               6502    35.96
not_humanitarian                         5228    28.91
rescue_volunteering_or_donation_effort   3775    20.88
infrastructure_and_utility_damage        1426     7.89
injured_or_dead_people                    533     2.95
affected_individuals                      516     2.85
vehicle_damage                             61     0.34
missing_or_found_people                    41     0.23


## Resolve duplicated images

Some tweets with different IDs may reuse the same image (e.g. retweets). We reconcile the labels for those shared images using the same frequency + confidence method as above.

In [15]:
# Show a few examples of duplicated images used in different tweets
print("Sample duplicated image hash entries (same image in different tweets):")
sample_img_dups = df[df["img_hash_str"].duplicated(keep=False)].sort_values("img_hash_str").head(10)
display(sample_img_dups[["img_hash_str", "tweet_id", "image_path", "image_info", "image_info_conf", "image_human", "image_human_conf","image_damage", "image_damage_conf"]])

Sample duplicated image hash entries (same image in different tweets):


,img_hash_str,tweet_id,image_path,image_info,image_info_conf,image_human,image_human_conf,image_damage,image_damage_conf
5840,8000000000000000,909777730924896256,data_image/hurricane_harvey/18_9_2017/90977773...,not_informative,0.6579,not_humanitarian,0.6579,NaN,NaN
8490,8000000000000000,909839409222230017,data_image/hurricane_irma/18_9_2017/9098394092...,not_informative,1.0000,not_humanitarian,1.0000,NaN,NaN
10102,8000000000000000,910197668680540160,data_image/hurricane_irma/19_9_2017/9101976686...,not_informative,1.0000,not_humanitarian,1.0000,NaN,NaN
5289,802b2deaca2e5e7a,908168392716234753,data_image/hurricane_harvey/14_9_2017/90816839...,informative,1.0000,infrastructure_and_utility_damage,1.0000,little_or_no_damage,0.6552
3823,802b2deaca2e5e7a,905900370412441600,data_image/hurricane_harvey/7_9_2017/905900370...,informative,0.6381,infrastructure_and_utility_damage,0.6381,mild_damage,0.6580
1919,8072fea0985a5ed7,901814144680255488,data_image/hurricane_harvey/27_8_2017/90181414...,informative,1.0000,infrastructure_and_utility_damage,1.0000,severe_damage,1.0000
1929,8072fea0985a5ed7,901822511750406144,data_image/hurricane_harvey/27_8_2017/90182251...,informative,0.6785,infrastructure_and_utility_damage,0.6785,severe_damage,0.6864
11934,80734de64ac55bad,913176642889109504,data_image/hurricane_maria/27_9_2017/913176642...,informative,0.6826,infrastructure_and_utility_damage,0.6826,mild_damage,0.7015
11904,80734de64ac55bad,913129211417829378,data_image/hurricane_maria/27_9_2017/913129211...,informative,1.0000,affected_individuals,1.0000,NaN,NaN
15496,80b4be4b5656956d,930457005550112770,data_image/iraq_iran_earthquake/14_11_2017/930...,informative,1.0000,infrastructure_and_utility_damage,1.0000,severe_damage,0.7823


In [16]:
# Mark only the second and subsequent occurrences of each img_hash_str as duplicate
df["image_dup"] = df.duplicated(subset=["img_hash_str"], keep="first")

In [17]:
hash_counts = df["img_hash_str"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1].index

# Find img_hash groups with inconsistent image labels
conflicting_img_hashes = (
	df[df["img_hash_str"].isin(duplicate_hashes)]
	.groupby("img_hash_str")[["image_info", "image_human", "image_damage"]]
	.nunique()
	.query("image_info > 1 or image_human > 1 or image_damage > 1")
	.index
)

print(f"⚠️ Image hashes with conflicting image labels: {len(conflicting_img_hashes)}")

# Show a few examples
df_conflicting_images = df[df["img_hash_str"].isin(conflicting_img_hashes)].sort_values("img_hash_str")
display(df_conflicting_images[["img_hash_str", "tweet_id", "image_path", "image_info", "image_info_conf", "image_human", "image_human_conf","image_damage", "image_damage_conf"]].head(10))


⚠️ Image hashes with conflicting image labels: 126


,img_hash_str,tweet_id,image_path,image_info,image_info_conf,image_human,image_human_conf,image_damage,image_damage_conf
3823,802b2deaca2e5e7a,905900370412441600,data_image/hurricane_harvey/7_9_2017/905900370...,informative,0.6381,infrastructure_and_utility_damage,0.6381,mild_damage,0.6580
5289,802b2deaca2e5e7a,908168392716234753,data_image/hurricane_harvey/14_9_2017/90816839...,informative,1.0000,infrastructure_and_utility_damage,1.0000,little_or_no_damage,0.6552
11904,80734de64ac55bad,913129211417829378,data_image/hurricane_maria/27_9_2017/913129211...,informative,1.0000,affected_individuals,1.0000,NaN,NaN
11934,80734de64ac55bad,913176642889109504,data_image/hurricane_maria/27_9_2017/913176642...,informative,0.6826,infrastructure_and_utility_damage,0.6826,mild_damage,0.7015
11382,80d47f43786a26eb,911982031898365952,data_image/hurricane_maria/24_9_2017/911982031...,informative,0.6637,other_relevant_information,0.6637,NaN,NaN
12508,80d47f43786a26eb,914886154402615296,data_image/hurricane_maria/2_10_2017/914886154...,not_informative,0.6486,not_humanitarian,0.6486,NaN,NaN
2181,81fde0865f4af0d8,904339363072544768,data_image/hurricane_harvey/3_9_2017/904339363...,informative,0.6451,vehicle_damage,0.6451,NaN,NaN
4332,81fde0865f4af0d8,906687728674332672,data_image/hurricane_harvey/10_9_2017/90668772...,informative,1.0000,infrastructure_and_utility_damage,1.0000,severe_damage,1.0000
17489,82ade3f4c574711a,872175891929063426,data_image/srilanka_floods/6_6_2017/8721758919...,informative,0.3696,affected_individuals,0.3696,NaN,NaN
17186,82ade3f4c574711a,870209884452552705,data_image/srilanka_floods/1_6_2017/8702098844...,informative,0.3939,infrastructure_and_utility_damage,0.3939,severe_damage,1.0000


In [18]:
print_class_distribution(df, "image_info")
print_class_distribution(df, "image_human")
print_class_distribution(df, "image_damage")


`image_info` distribution:
                 count  percent
image_info                     
informative       9374    51.84
not_informative   8708    48.16

`image_human` distribution:
                                        count  percent
image_human                                           
not_humanitarian                         8708    48.16
infrastructure_and_utility_damage        3624    20.04
other_relevant_information               2529    13.99
rescue_volunteering_or_donation_effort   2231    12.34
affected_individuals                      562     3.11
vehicle_damage                            304     1.68
injured_or_dead_people                    110     0.61
missing_or_found_people                    14     0.08

`image_damage` distribution:
                         count  percent
image_damage                           
NaN                      14455    79.94
severe_damage             2212    12.23
mild_damage                839     4.64
little_or_no_damage        475     

In [19]:
# Count NaN values in "image_damage" and "image_damage_conf"
image_damage_nan = df["image_damage"].isna().sum()
image_damage_conf_nan = df["image_damage_conf"].isna().sum()

print(f"Total NaN values in image_damage: {image_damage_nan}")
print(f"Total NaN values in image_damage_conf: {image_damage_conf_nan}")

Total NaN values in image_damage: 14455
Total NaN values in image_damage_conf: 14455


In [20]:
# Fill NaN values in 'image_damage' with "unknown"
df["image_damage"] = df["image_damage"].fillna("unknown")

# Fill NaN values in 'image_damage_conf' with 0.0
df["image_damage_conf"] = df["image_damage_conf"].fillna(0.01)

# Print updated counts
image_damage_nan = df["image_damage"].isna().sum()
unknown_count = (df["image_damage"] == "unknown").sum()
image_damage_conf_nan = df["image_damage_conf"].isna().sum()
zero_conf_count = (df["image_damage_conf"] == 0.01).sum()

print(f"Total NaN values in image_damage: {image_damage_nan}")
print(f"Total 'unknown' values in image_damage: {unknown_count}\n")
print(f"Total NaN values in image_damage_conf: {image_damage_conf_nan}")
print(f"Total 0.01 values in image_damage_conf: {zero_conf_count}\n")


Total NaN values in image_damage: 0
Total 'unknown' values in image_damage: 14455

Total NaN values in image_damage_conf: 0
Total 0.01 values in image_damage_conf: 14455



In [21]:
for label_col in ["image_info", "image_human", "image_damage"]:
    conf_col = label_col + "_conf"
    reconciled = (
        duplicated_tweets.groupby("img_hash_str", group_keys=False)
        .apply(lambda g: reconcile_with_confidence(g, label_col, conf_col, alpha=0.6), include_groups=False)
    )
    # Map reconciled labels back into the full dataframe
    df.loc[df["img_hash_str"].isin(reconciled.index), label_col] = df["img_hash_str"].map(reconciled)

In [22]:
# Image hash duplication stats
n_image_dups = df["image_dup"].sum()  # Only counts second and subsequent occurrences
n_unique_img_hash_groups = len(conflicting_img_hashes)

print(f"Total duplicated image hash rows (excluding first occurrence): {n_image_dups}")
print(f"Unique image hash groups reconciled: {n_unique_img_hash_groups}")

Total duplicated image hash rows (excluding first occurrence): 728
Unique image hash groups reconciled: 126


In [23]:
print_class_distribution(df, "image_info")
print_class_distribution(df, "image_human")
print_class_distribution(df, "image_damage")


`image_info` distribution:
                 count  percent
image_info                     
informative       9368    51.81
not_informative   8714    48.19

`image_human` distribution:
                                        count  percent
image_human                                           
not_humanitarian                         8714    48.19
infrastructure_and_utility_damage        3621    20.03
other_relevant_information               2526    13.97
rescue_volunteering_or_donation_effort   2230    12.33
affected_individuals                      562     3.11
vehicle_damage                            305     1.69
injured_or_dead_people                    110     0.61
missing_or_found_people                    14     0.08

`image_damage` distribution:
                         count  percent
image_damage                           
unknown                  11894    65.78
NaN                       2562    14.17
severe_damage             2214    12.24
mild_damage                838     

## Aggregate low-frequency humanitarian categories

To reduce class imbalance and sparsity in the `*_human` labels, we consolidate the following rare classes:
- `injured_or_dead_people` + `missing_or_found_people` → `affected_individuals`
- `vehicle_damage` → `infrastructure_and_utility_damage`

This improves class distribution and model generalization.

In [24]:
# Merge small humanitarian categories into broader classes
merge_map = {
    "injured_or_dead_people": "affected_individuals",
    "missing_or_found_people": "affected_individuals",
    "vehicle_damage": "infrastructure_and_utility_damage"
}

df["text_human"] = df["text_human"].replace(merge_map)
df["image_human"] = df["image_human"].replace(merge_map)

In [25]:
print_class_distribution(df, "text_human")


`text_human` distribution:
                                        count  percent
text_human                                            
other_relevant_information               6502    35.96
not_humanitarian                         5228    28.91
rescue_volunteering_or_donation_effort   3775    20.88
infrastructure_and_utility_damage        1487     8.22
affected_individuals                     1090     6.03


In [26]:
print_class_distribution(df, "image_human")


`image_human` distribution:
                                        count  percent
image_human                                           
not_humanitarian                         8714    48.19
infrastructure_and_utility_damage        3926    21.71
other_relevant_information               2526    13.97
rescue_volunteering_or_donation_effort   2230    12.33
affected_individuals                      686     3.79


## Multimodal label fusion

We define new target variables that integrate both text and image views:

- `mm_info`: set to `"informative"` if **either** `text_info` or `image_info` is informative
- `mm_human`: we apply a rule-based strategy


In [27]:
def fuse_info(row):
	if row["text_info"] == "informative" or row["image_info"] == "informative":
		return "informative"
	return "not_informative"

df["mm_info"] = df.apply(fuse_info, axis=1)

In [28]:
def fuse_human(row):
    t_info, i_info = row["text_info"], row["image_info"]
    t_hum, i_hum = row["text_human"], row["image_human"]

    # Rule 1: if one modality is not informative, take human label from the informative one
    if t_info == "not informative" and i_info == "informative":
        return i_hum
    if i_info == "not informative" and t_info == "informative":
        return t_hum

    # Rule 2: if one of the two human labels is 'not_humanitarian', return the other
    if t_hum == "not_humanitarian" and i_hum != "not_humanitarian":
        return i_hum
    if i_hum == "not_humanitarian" and t_hum != "not_humanitarian":
        return t_hum

	# Rule 3: if one of the two human labels is 'other_relevant_information', return the other
    if t_hum == "other_relevant_information" and i_hum != "other_relevant_information":
        return i_hum
    if i_hum == "other_relevant_information" and t_hum != "other_relevant_information":
        return t_hum

    # Rule 4: if both labels are the same, return it
    if t_hum == i_hum:
        return t_hum

    # Rule 5: unresolved case
    return "conflict"

df["mm_human"] = df.apply(fuse_human, axis=1)


In [29]:
print_class_distribution(df, "mm_info")
print_class_distribution(df, "mm_human")


`mm_info` distribution:
                 count  percent
mm_info                        
informative      13797     76.3
not_informative   4285     23.7

`mm_human` distribution:
                                        count  percent
mm_human                                              
not_humanitarian                         4285    23.70
other_relevant_information               4209    23.28
rescue_volunteering_or_donation_effort   3821    21.13
infrastructure_and_utility_damage        3438    19.01
conflict                                 1450     8.02
affected_individuals                      879     4.86


In [30]:
# Show examples with mm_human == 'conflict'
conflict_examples = df[df["mm_human"] == "conflict"]

print(f"⚠️ Found {len(conflict_examples)} entries with mm_human = 'conflict'")
display(conflict_examples[
    ["tweet_id", "tweet_text", "image_path", "text_info", "image_info", "text_human", "image_human", "mm_human"]
].head(10))


⚠️ Found 1450 entries with mm_human = 'conflict'


,tweet_id,tweet_text,image_path,text_info,image_info,text_human,image_human,mm_human
1,917791130590183424,PHOTOS: Deadly wildfires rage in California ht...,data_image/california_wildfires/10_10_2017/917...,informative,informative,infrastructure_and_utility_damage,affected_individuals,conflict
6,917792930315821057,Mass Evacuations in California as Wildfires Ki...,data_image/california_wildfires/10_10_2017/917...,informative,informative,affected_individuals,infrastructure_and_utility_damage,conflict
12,917793881533571073,Wildfires Still Burn in Northern California; 1...,data_image/california_wildfires/10_10_2017/917...,informative,informative,affected_individuals,infrastructure_and_utility_damage,conflict
14,917794232160661505,At Least 11 Dead and 100 Missing as Wildfires ...,data_image/california_wildfires/10_10_2017/917...,informative,informative,affected_individuals,infrastructure_and_utility_damage,conflict
15,917794360581869569,Southern California wildfires continue to rage...,data_image/california_wildfires/10_10_2017/917...,informative,informative,affected_individuals,infrastructure_and_utility_damage,conflict
16,917794580728295424,More than 100 missing persons reports made in ...,data_image/california_wildfires/10_10_2017/917...,informative,informative,affected_individuals,infrastructure_and_utility_damage,conflict
17,917794892113498113,"RT @News12BX: California wildfires kill 10, de...",data_image/california_wildfires/10_10_2017/917...,informative,informative,affected_individuals,infrastructure_and_utility_damage,conflict
19,917795236595863552,"11 dead, thousands homeless as wildfires torch...",data_image/california_wildfires/10_10_2017/917...,informative,informative,affected_individuals,infrastructure_and_utility_damage,conflict
20,917796280377602048,Thousands flee as wildfires ravage northern Ca...,data_image/california_wildfires/10_10_2017/917...,informative,informative,affected_individuals,rescue_volunteering_or_donation_effort,conflict
23,917796940900982786,How to help Napa fire victims: 8 things you ca...,data_image/california_wildfires/10_10_2017/917...,informative,informative,rescue_volunteering_or_donation_effort,infrastructure_and_utility_damage,conflict


## Text Cleaning and Normalization

We apply a set of light preprocessing steps to clean the tweet text while preserving its semantic content. This helps standardize input across samples and reduces noise:

- **Removing URLs, usernames, and hashtags**
- **Unescaping HTML entities** (e.g., `&amp;` → `&`)
- **Stripping emojis and excessive punctuation**
- **Removing accents and diacritics** using Unicode normalization

The goal is to clean the data without losing informative tokens that may be useful for classification (e.g., "rescue", "help", "damage").



In [31]:
pd.set_option('display.max_colwidth', None)
df["tweet_text"].head(10)

0             RT @Gizmodo: Wildfires raging through Northern California are terrifying https://t.co/dI73RFzX2i https://t.co/k4KnvIimsU
1                                          PHOTOS: Deadly wildfires rage in California https://t.co/td9xT3vXOL https://t.co/OimwAncLew
2       RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax
3       RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax
4             RT @TIME: California's raging wildfires as you've never seen them before https://t.co/OksQOZ2LHH https://t.co/oHTMbrM2Jx
5                         Wildfires Threaten California's First Legal Cannabis Harvest https://t.co/BSuAdfwN62 https://t.co/wNqfLrNmTp
6    Mass Evacuations in California as Wildfires Kill at Least 10 https://t.co/gyoKFWZuMB #CaliforniaWildfires https://t.co/KEFtjITetK
7        RT @KAKEnews: California wildfires destroy mor

In [32]:
def preprocess_tweet(text, model_type="embedding"):
	"""
	Preprocess a tweet based on the target model type.

	Parameters:
		text (str): The raw tweet text.
		model_type (str): One of ["embedding", "classic", "bertweet"].
			- "bertweet": minimal cleaning, tailored for BERTweet tokenizer
			- "embedding": general BERT/RoBERTa-style embedding models

	Returns:
		str: Preprocessed tweet text.
	"""
	# Fix text encoding issues
	text = html.unescape(text)
	text = unicodedata.normalize("NFKD", text)

	# Remove leading retweet marker "RT @user:"
	text = re.sub(r'^RT\s+@[\w_]+:\s+', '', text)

	# --- BERTweet-specific preprocessing ---
	if model_type == "bertweet":
		# DO NOT remove URLs, mentions, emojis
		# Just clean up retweet and encoding
		return text.strip()

	# --- For all other models ---
	# Replace URLs with placeholder
	text = re.sub(r'http\S+', '', text)
	
	# Replace mentions with placeholder
	text = re.sub(r'@\w+', '', text)

	# Convert emojis to text (e.g. 😢 → :crying_face:)
	text = emoji.demojize(text, delimiters=(" ", " "))

	# Remove '#' from hashtags, keep the word
	text = re.sub(r'#', '', text)

	# Normalize whitespace
	text = re.sub(r'\s+', ' ', text).strip()

	return text

In [33]:
# Apply preprocessing and create new columns for each model
df['cleaned_text_bert'] = df['tweet_text'].apply(lambda x: preprocess_tweet(x, model_type='embedding'))
df['cleaned_text_bertweet'] = df['tweet_text'].apply(lambda x: preprocess_tweet(x, model_type='bertweet'))


In [34]:
df[['tweet_text', 'cleaned_text_bert','cleaned_text_bertweet'] ].head(5)

,tweet_text,cleaned_text_bert,cleaned_text_bertweet
0,RT @Gizmodo: Wildfires raging through Northern California are terrifying https://t.co/dI73RFzX2i https://t.co/k4KnvIimsU,Wildfires raging through Northern California are terrifying,Wildfires raging through Northern California are terrifying https://t.co/dI73RFzX2i https://t.co/k4KnvIimsU
1,PHOTOS: Deadly wildfires rage in California https://t.co/td9xT3vXOL https://t.co/OimwAncLew,PHOTOS: Deadly wildfires rage in California,PHOTOS: Deadly wildfires rage in California https://t.co/td9xT3vXOL https://t.co/OimwAncLew
2,"RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax","PLS SHARE: We're capturing wildfire response, recovery info here:","PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax"
3,"RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax","PLS SHARE: We're capturing wildfire response, recovery info here:","PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax"
4,RT @TIME: California's raging wildfires as you've never seen them before https://t.co/OksQOZ2LHH https://t.co/oHTMbrM2Jx,California's raging wildfires as you've never seen them before,California's raging wildfires as you've never seen them before https://t.co/OksQOZ2LHH https://t.co/oHTMbrM2Jx


## Save cleaned dataset

In [35]:
save_path = os.path.join(data_dir, "preprocessed_annotations.tsv")

df.to_csv(save_path, sep="\t", index=False)